# Per-strain activity models

For each of the five bacterial strains, four classifiers (MLP, SVM, RF, XGBoost) are compared by 10-fold cross-validation at predicting activity against that strain.

#### Imports

In [1]:
import numpy as np
import pandas as pd
from random import seed, sample

from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

#### Configuration

In [ ]:
def seq_fingerprint(sequence):
    fp = np.zeros(len(sequence))
    for i, char in enumerate(sequence):
        if char.isupper():
            fp[i] = 1
    return fp
strains = {
    'E. coli W3110':          'data/activity-ecoli.xlsx',
    'P. aeruginosa PAO1':     'data/activity-paerug.xlsx',
    'A. baumannii ATCC19606': 'data/activity-abaum.xlsx',
    'K. pneumoniae NCTC418':  'data/activity-kpneu.xlsx',
    'MRSA (S. aureus COL)':   'data/activity-mrsa.xlsx',}

mlp_alpha = {
    'E. coli W3110': 1,
    'P. aeruginosa PAO1': 0.1,
    'A. baumannii ATCC19606': 1,
    'K. pneumoniae NCTC418': 1,
    'MRSA (S. aureus COL)': 1,}

def make_models(alpha):
    return {
        'MLP':     lambda: MLPClassifier(alpha=alpha, max_iter=1000, hidden_layer_sizes=(32, 32, 32), random_state=42),
        'SVM':     lambda: SVC(kernel='rbf', probability=True, random_state=42),
        'RF':      lambda: RandomForestClassifier(n_estimators=100, random_state=42),
        'XGBoost': lambda: XGBClassifier(n_estimators=100, eval_metric='logloss', random_state=42)}

#### Load and preprocess each strain

In [ ]:
def load_strain(path):
    df = pd.read_excel(path)
    label_col = [c for c in df.columns if c not in ('Name', 'Sequence', 'round')][0]
    df = df[df['round'] <= 4].reset_index(drop=True)
    X = np.array([seq_fingerprint(s) for s in df['Sequence']])
    y = np.array(df[label_col].astype(int).tolist())
    return X, y
data = {}
for sname, path in strains.items():
    X, y = load_strain(path)
    data[sname] = (X, y)
    print(sname)

E. coli W3110
P. aeruginosa PAO1
A. baumannii ATCC19606
K. pneumoniae NCTC418
MRSA (S. aureus COL)


#### Cross-validation helper

In [4]:
def cv_metrics(factory, X, y, cv):
    aucs, accs, precs, recs, f1s = [], [], [], [], []
    for train, test in cv.split(X, y):
        model = factory()
        model.fit(X[train], y[train])
        score = model.predict_proba(X[test])[:, 1]
        pred = model.predict(X[test])
        aucs.append(roc_auc_score(y[test], score))
        accs.append(accuracy_score(y[test], pred))
        precs.append(precision_score(y[test], pred, zero_division=0))
        recs.append(recall_score(y[test], pred, zero_division=0))
        f1s.append(f1_score(y[test], pred, zero_division=0))
    return {'AUC': np.mean(aucs), 'AUC_std': np.std(aucs),
            'accuracy': np.mean(accs), 'precision': np.mean(precs),
            'recall': np.mean(recs), 'F1': np.mean(f1s)}

#### Run the strain: model comparison (true and scrambled labels)

In [5]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

rows = []
for sname, (X, y) in data.items():
    seed(42)
    y_scrambled = np.array(sample(list(y), len(y)))
    for mname, factory in make_models(mlp_alpha[sname]).items():
        m = cv_metrics(factory, X, y, cv)
        auc_scr = cv_metrics(factory, X, y_scrambled, cv)['AUC']
        rows.append({'strain': sname, 'model': mname,
                     'AUC': round(m['AUC'], 2), 'AUC_std': round(m['AUC_std'], 2),
                     'AUC_scrambled': round(auc_scr, 2),
                     'accuracy': round(m['accuracy'], 2), 'precision': round(m['precision'], 2),
                     'recall': round(m['recall'], 2), 'F1': round(m['F1'], 2)})

results = pd.DataFrame(rows)
results

,strain,model,AUC,AUC_std,AUC_scrambled,accuracy,precision,recall,F1
0,E. coli W3110,MLP,0.85,0.06,0.57,0.84,0.87,0.94,0.90
1,E. coli W3110,SVM,0.80,0.07,0.55,0.80,0.80,1.00,0.89
2,E. coli W3110,RF,0.73,0.09,0.52,0.79,0.81,0.96,0.88
3,E. coli W3110,XGBoost,0.77,0.08,0.55,0.80,0.84,0.93,0.88
4,P. aeruginosa PAO1,MLP,0.77,0.08,0.54,0.69,0.69,0.66,0.67
5,P. aeruginosa PAO1,SVM,0.80,0.09,0.55,0.77,0.79,0.70,0.74
6,P. aeruginosa PAO1,RF,0.78,0.09,0.53,0.69,0.67,0.71,0.69
7,P. aeruginosa PAO1,XGBoost,0.77,0.09,0.60,0.70,0.68,0.66,0.67
8,A. baumannii ATCC19606,MLP,0.92,0.06,0.53,0.84,0.86,0.84,0.85
9,A. baumannii ATCC19606,SVM,0.89,0.06,0.60,0.79,0.83,0.80,0.81


#### AUC summary

Saved to `output/bacteria_model_comparison.csv`.

In [6]:
auc_table = results.pivot(index='strain', columns='model', values='AUC')[['MLP', 'SVM', 'RF', 'XGBoost']]
auc_table = auc_table.reindex(list(strains.keys()))
results.to_csv('output/bacteria_model_comparison.csv', index=False)
auc_table

model,MLP,SVM,RF,XGBoost
strain,,,,
E. coli W3110,0.85,0.80,0.73,0.77
P. aeruginosa PAO1,0.77,0.80,0.78,0.77
A. baumannii ATCC19606,0.92,0.89,0.91,0.90
K. pneumoniae NCTC418,0.89,0.87,0.85,0.82
MRSA (S. aureus COL),0.88,0.85,0.86,0.84
